In [12]:
import os
import re
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import requests
from io import BytesIO
import pandas as pd

In [4]:
# Load the BLIP model and processor from Hugging Face, do this once at the start
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [ ]:
def generate_image_captions(image_folder, sample_size=50):
    """
    Takes a folder of images, generates captions for a random sample, and returns a list of captions.
    """
    captions = []
    image_files = os.listdir(image_folder)[:sample_size]  # Limit to sample size
    
    for file in image_files:
        #print(file)
        raw_image = Image.open(os.path.join(image_folder, file)).convert("RGB")
        
        inputs = processor(raw_image, return_tensors="pt")
        out = model.generate(**inputs)
        caption = processor.decode(out[0], skip_special_tokens=True)
        
        captions.append((caption, file))
        
    return captions

In [8]:
test = generate_image_captions("./e-commerce/Apparel/Boys/Images/images_with_product_ids")
print(test)

4203.jpg
38491.jpg
24906.jpg
24912.jpg
36325.jpg
39002.jpg
36319.jpg
39758.jpg
34056.jpg
34042.jpg
31112.jpg
40933.jpg
31106.jpg
40927.jpg
5486.jpg
31099.jpg
3815.jpg
36721.jpg
38283.jpg
39837.jpg
41754.jpg
41967.jpg
37173.jpg
37167.jpg
37601.jpg
37600.jpg
37172.jpg
35995.jpg
41966.jpg
38282.jpg
24873.jpg
37199.jpg
36720.jpg
3814.jpg
31098.jpg
40926.jpg
31107.jpg
31113.jpg
34043.jpg
34057.jpg
46833.jpg
39759.jpg
33260.jpg
36318.jpg
39003.jpg
36324.jpg
24913.jpg
38490.jpg
24907.jpg
4202.jpg
[("a boy ' s blue striped t - shirt with a sail and a sail", '4203.jpg'), ('t - shirt with print', '38491.jpg'), ('t - shirt with print', '24906.jpg'), ('t - shirt with cat print', '24912.jpg'), ('t - shirt be happy', '36325.jpg'), ('a black t - shirt with a green star on the front', '39002.jpg'), ('t - shirt with a cartoon character', '36319.jpg'), ('a picture of a grey cargo shorts', '39758.jpg'), ('a picture of a denim shorts with a skull on the side', '34056.jpg'), ('a red and white shirt with a 

In [9]:
# for handling URLs instead of local files:
def generate_image_url_captions(df, url_column, sample_size=50):
    """
    Takes a pandas DataFrame and a column of image URLs, fetches a random sample,
    and returns a list of generated captions.
    """
    captions = []
    valid_df = df.dropna(subset=[url_column])
    sample_df = valid_df.sample(n=min(sample_size, len(valid_df)))
    
    for index, row in sample_df.iterrows():
        url = row[url_column]
        
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            
            raw_image = Image.open(BytesIO(response.content)).convert('RGB')
            
            inputs = processor(raw_image, return_tensors="pt")
            out = model.generate(**inputs)
            caption = processor.decode(out[0], skip_special_tokens=True)
            
            captions.append((caption, url))
        except Exception as e:
            print(f"Skipped image at index {index} due to error: {e}")
            continue
        
    return captions

In [15]:
# example usage:
amazon_df = pd.read_csv("./amazon.csv")
captions = generate_image_url_captions(df=amazon_df, url_column="img_link", sample_size=10)

Skipped image at index 1006 due to error: 400 Client Error: Bad Request for url: https://m.media-amazon.com/images/W/WEBP_402378-T1/images/I/51ca6eZ+j3L._SY300_SX300_.jpg
Skipped image at index 931 due to error: 400 Client Error: Bad Request for url: https://m.media-amazon.com/images/W/WEBP_402378-T2/images/I/51HO3bkK+VS._SY300_SX300_.jpg


In [16]:
print(captions)

[('a black and white wine bottle with a black handle', 'https://m.media-amazon.com/images/I/21OPu5-M3qL._SX300_SY300_QL70_FMwebp_.jpg'), ('a ceiling fan with a black blade', 'https://m.media-amazon.com/images/I/21qojQDoKWL._SX300_SY300_QL70_FMwebp_.jpg'), ('a blender with a blender and blender attachment', 'https://m.media-amazon.com/images/I/41yPeG8kXxL._SX300_SY300_QL70_FMwebp_.jpg'), ('a usb cable with a usb cable attached to it', 'https://m.media-amazon.com/images/I/31l-eZHBfKL._SX300_SY300_QL70_FMwebp_.jpg'), ('a black mixer with a bowl and a mixer attachment', 'https://m.media-amazon.com/images/I/41hYZPZaWfS._SX300_SY300_QL70_FMwebp_.jpg'), ('the apple watch is shown in black', 'https://m.media-amazon.com/images/I/41u0PC4NajL._SX300_SY300_QL70_ML2_.jpg'), ('caso caso caso caso caso caso caso caso caso caso', 'https://m.media-amazon.com/images/I/41LcHKyVl9L._SX300_SY300_QL70_FMwebp_.jpg'), ('anker usb cable usb usb usb usb usb usb usb usb usb usb usb usb usb usb usb usb', 'https:/

### next steps: 
##### test other image caption models
##### build content and semantic profiler
##### deal with text
##### automatically identify image columns

### Automatic column type detection

In [21]:
def detect_modalities(df):
    """
    Analyzes a pandas DataFrame to detect which columns likely contain tabular data, text, or image URLs.
    """
    modality_map = {
        'tabular': [],
        'text': [],
        'image_url': []
    }
    image_url_pattern = re.compile(r'^https?://.*\.(?:jpg|jpeg|png|gif|webp).*$', re.IGNORECASE)
    
    for column in df.columns:
        if df[column].dtype == 'object':
            valid_sample = df[column].dropna().astype(str)
            if len(valid_sample) == 0:
                continue
            
            if valid_sample.str.match(image_url_pattern).mean() > 0.5:
                modality_map['image_url'].append(column)
                
            elif valid_sample.str.len().mean() > 50:
                modality_map['text'].append(column)
            else:
                modality_map['tabular'].append(column)
        else:
            modality_map['tabular'].append(column)
            
    return modality_map

In [22]:
modality_info = detect_modalities(amazon_df)
print(modality_info)

{'tabular': ['product_id', 'discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count'], 'text': ['product_name', 'category', 'about_product', 'user_id', 'user_name', 'review_id', 'review_title', 'review_content', 'product_link'], 'image_url': ['img_link']}


### Automatic image file detection

In [23]:
def scan_dataset_directory(base_path):
    """
    Scans a directory to identify csvs and image folders
    """
    dataset_assets = {
        'csv_files': [],
        'image_folders': []
    }
    valid_image_extensions = ('.png', '.jpg', '.jpeg')
    
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file.endswith('.csv'):
                dataset_assets['csv_files'].append(os.path.join(root, file))
                
            elif file.lower().endswith(valid_image_extensions):
                if root not in dataset_assets['image_folders']:
                    dataset_assets['image_folders'].append(root)
                    
    return dataset_assets

In [24]:
e_commerce_assets = scan_dataset_directory("./e-commerce")
print(e_commerce_assets)

{'csv_files': ['./e-commerce/fashion.csv'], 'image_folders': ['./e-commerce/Footwear/Men/Images/images_with_product_ids', './e-commerce/Footwear/Women/Images/images_with_product_ids', './e-commerce/Apparel/Girls/Images/images_with_product_ids', './e-commerce/Apparel/Boys/Images/images_with_product_ids']}
